<a href="https://colab.research.google.com/github/himanshusar123/-Machine-Learning-Quiz-Classification-or-Regression-/blob/main/Day_25_Harbinger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 25: Harbinger - NLP & LLM Fundamentals

This notebook explores fundamental operations in natural language processing (NLP) and Large Language Model (LLM) workflows:
1. **Tokenization** using OpenAI's `tiktoken` library.
2. **Zero-Shot Customer Feedback Classification** using Groq Cloud API.
3. **SmartSupport AI**: A mini project implementing structured JSON output extraction for support tickets.

## Part 1: Sub-word Tokenization with `tiktoken`

### What is Tokenization?
Tokenization is the process of translating raw text into integers (tokens) that LLMs can process. Modern LLMs use **Byte Pair Encoding (BPE)**, which breaks words down into common sub-word patterns to handle spelling variations, rare vocabulary, and multi-language inputs efficiently.

### What is `cl100k_base`?
This is the specific vocabulary encoding ruleset used by OpenAI's `gpt-4` and `gpt-3.5-turbo` models.

In [2]:
import tiktoken

# Load the cl100k_base encoding scheme (used by GPT-4 and GPT-3.5-turbo)
encoding = tiktoken.get_encoding("cl100k_base")

# Define the sample text to tokenize
text = "Artificial Intelligence is transforming businesses."

# Encode the text into a list of token IDs
tokens = encoding.encode(text)

# Print the list of token IDs and the total token count
print("Token IDs:", tokens)
print("Number of tokens:", len(tokens))

[9470, 16895, 22107, 374, 46890, 9873, 13]
Number of tokens: 7


## Part 2: Zero-Shot Text Classification via Groq API

### What is Zero-Shot Classification?
Instead of training a custom Machine Learning classifier (like SVM or Logistic Regression), we instruct a pre-trained Large Language Model (LLM) to categorize customer messages. It requires no training data, only a clear instruction (prompt).

In [3]:
# Step 1: Install required libraries
!pip install groq

# Step 2: Import required libraries
import os
from groq import Groq
from google.colab import userdata

# Step 3: Load API key safely from Colab secrets
# Make sure you have added "GROQ_API_KEY" to your Colab secrets (left sidebar key icon)
groq_api_key = userdata.get("GROQ_API_KEY")

# Step 4: Initialize the Groq client
client = Groq(api_key=groq_api_key)

# Step 5: Define the prompt and create the chat completion
# We pass a multi-line instruction prompting the model to extract Sentiment, Issue, and Priority.
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # High-performance Llama-3 model hosted on Groq
    messages=[
        {
            "role": "user",
            "content": """
            Classify the following customer feedback.

            Feedback:
            The application is easy to use but frequently crashes.

            Return:
            Sentiment
            Issue
            Priority
            """
        }
    ]
)

# Step 6: Print output generated by the LLM
print(response.choices[0].message.content)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.8 MB/s eta 0:00:00
Based on the customer feedback, here is the classification:

* **Sentiment**: Neutral (the customer mentions both a positive aspect - "easy to use" - and a negative aspect - "frequently crashes")
* **Issue**: Technical Issue (the application crashes, which is a technical problem that needs to be resolved)
* **Priority**: High (frequent crashes can be frustrating and prevent the customer from using the application effectively, making it a high-priority issue to fix)


## Part 3: Mini Project: SmartSupport AI – Customer Ticket Analyzer

### Scenario
ShopEase, an e-commerce company, receives hundreds of customer-support messages every day.

**Examples:**
- *"I ordered a laptop yesterday and it arrived with a cracked screen. I want a refund now!"*
- *"Where is my order? It was supposed to arrive three days ago."*
- *"The application is easy to use but frequently crashes."*

Currently, support team members manually read and categorize each ticket's sentiment, category, and urgency. Your task is to build a generative AI helper that automatically processes these unstructured text blocks into structured data objects.

### Expected Output Format

For input text:
> *"The application is easy to use but frequently crashes."*

The model must return structured JSON format:
```json
{
    "sentiment": "negative",
    "issue": "application crash",
    "category": "technical",
    "priority": "high"
}
```

In [7]:
import json
# Install the groq SDK
!pip install groq

from groq import Groq
from google.colab import userdata

# Initialize Groq client with API key stored securely in Google Colab Secrets
client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

# Define the incoming customer ticket text
feedback = """
I ordered a laptop yesterday.
It arrived with a cracked screen.
I want a refund immediately.
"""

# Request structured Chat Completion from Groq
# Use system instructions to strictly constrain the model to output ONLY valid JSON format.
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": """
You are a customer support ticket classification system.

Analyze customer feedback and return:

sentiment
issue
category
priority

Do not invent information that is not provided.

Return ONLY valid JSON.
"""
        },
        {
            "role": "user",
            "content": feedback
        }
    ],
    # Low temperature guarantees consistent and highly reproducible outputs
    temperature=0.1
)

# Extract and print result
result = response.choices[0].message.content
print(result)

```
{
  "sentiment": "negative",
  "issue": "damaged product",
  "category": "returns and refunds",
  "priority": "high"
}
```
